# 04 — Fine-Tuning con LoRA: Proceso Completo

Este notebook visualiza el proceso de fine-tuning LoRA para especializar
un modelo de lenguaje en el dominio del Z2H-Shop Assistant.

## ¿Qué es LoRA?

**LoRA (Low-Rank Adaptation)** es una técnica de fine-tuning eficiente que:

- **No modifica los pesos originales** del modelo base
- Inyecta matrices de adaptación de **bajo rank** en capas clave (atención)
- Entrena solo **0.1-1%** de los parámetros totales
- Produce un **adaptador** pequeño (~10-100 MB) que se carga sobre el modelo base

### ¿Por qué LoRA?

| Método | Parámetros entrenados | VRAM necesaria | Calidad |
|--------|----------------------|----------------|---------|
| Fine-tuning completo | 100% | Muy alta (40GB+) | Alta |
| LoRA (r=16) | ~0.5% | Media (8-16GB) | Alta |
| QLoRA (r=16, 4-bit) | ~0.5% | Baja (4-8GB) | Alta |

LoRA es ideal cuando:
- Tienes datos de dominio específicos (tickets, conversaciones)
- Necesitas especializar el tono o formato de respuesta
- No tienes GPUs de alto rendimiento
- Quieres versionar diferentes adaptadores para diferentes tareas

## Setup e importaciones

In [ ]:
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from dotenv import load_dotenv

load_dotenv()

BASE_DIR = Path().resolve().parent
DATASET_PATH = BASE_DIR / "gold" / "finetune" / "dataset.jsonl"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Explorar el dataset de fine-tuning

Antes de entrenar, examinamos los datos que usaremos.

In [ ]:
if DATASET_PATH.exists():
    records = []
    with open(DATASET_PATH, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    df = pd.DataFrame(records)
    print(f"Total de registros: {len(df)}")
    print(f"\nColumnas: {list(df.columns)}")
    print(f"\nPrimeros 3 registros:")
    for i, row in df.head(3).iterrows():
        print(f"\n--- Registro {i+1} ---")
        print(f"Instruction: {row['instruction'][:80]}...")
        print(f"Input: {row['input'][:100]}...")
        print(f"Output: {row['output'][:100]}...")
        print(f"Longitud output: {len(row['output'])} chars")

    # Estadísticas de longitudes
    df["input_len"] = df["input"].str.len()
    df["output_len"] = df["output"].str.len()
    print(f"\nLongitud promedio input:  {df['input_len'].mean():.0f} chars")
    print(f"Longitud promedio output: {df['output_len'].mean():.0f} chars")
else:
    print(f"Dataset no encontrado en {DATASET_PATH}")
    print("Ejecuta build_finetune_dataset.py primero.")

In [ ]:
# Distribución de tipos de instrucción
if DATASET_PATH.exists():
    instruction_counts = df["instruction"].value_counts()
    print("Distribución por tipo de instrucción:")
    for inst, count in instruction_counts.items():
        print(f"  {count:4d} × {inst[:60]}...")

    # Gráfico de distribución
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Distribución de tipos
    instruction_counts.plot(kind="barh", ax=axes[0], color="steelblue")
    axes[0].set_title("Distribución por tipo de instrucción")
    axes[0].set_xlabel("Cantidad")

    # Distribución de longitudes de output
    axes[1].hist(df["output_len"], bins=30, color="coral", edgecolor="black")
    axes[1].set_title("Distribución de longitudes de output")
    axes[1].set_xlabel("Longitud (chars)")
    axes[1].set_ylabel("Frecuencia")
    axes[1].axvline(df["output_len"].mean(), color="red", linestyle="--",
                     label=f'Media: {df["output_len"].mean():.0f}')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## 2. Configurar el modelo y LoRA

### Parámetros de LoRA

- **r** (rank): Dimensión de las matrices de adaptación. Mayor r = más capacidad pero más VRAM.
- **alpha**: Factor de escala. Generalmente alpha = 2*r.
- **target_modules**: Capas donde se inyecta LoRA. `q_proj` y `v_proj` son las capas de atención.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_BASE = os.getenv("MODEL_BASE", "microsoft/phi-2")
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN", None)
LORA_R = int(os.getenv("LORA_R", "16"))
LORA_ALPHA = int(os.getenv("LORA_ALPHA", "32"))
LORA_DROPOUT = float(os.getenv("LORA_DROPOUT", "0.05"))

print(f"Modelo base: {MODEL_BASE}")
print(f"LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Target modules: ['q_proj', 'v_proj']")

In [ ]:
# Cargar modelo base
print(f"Cargando {MODEL_BASE}...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_BASE, token=HUGGINGFACE_TOKEN, trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    token=HUGGINGFACE_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Modelo cargado: {model.num_parameters() / 1e9:.2f}B parámetros")

In [ ]:
# Aplicar LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)

# Parámetros entrenables vs totales
model.print_trainable_parameters()

## 3. Tokenizar el dataset

In [ ]:
from datasets import Dataset

MAX_SEQ_LENGTH = int(os.getenv("MAX_SEQ_LENGTH", "512"))

def format_chat(example):
    return (
        f"<|system|\nEres un agente de soporte del Z2H-Shop.\n"
        f"<|user|\n{example['instruction']}\n\n{example['input']}\n"
        f"<|assistant|\n{example['output']}"
    )

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )

# Formatear y tokenizar
formatted = [format_chat(r) for r in records]
ds = Dataset.from_dict({"text": formatted})

# Split 90/10
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds = split["test"]

print(f"Train: {len(train_ds)} registros")
print(f"Validation: {len(val_ds)} registros")

# Tokenizar
train_tokenized = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tokenized = val_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

print(f"\nEjemplo tokenizado:")
print(f"  Input IDs shape: {train_tokenized[0]['input_ids']}")

## 4. Entrenamiento

Configuramos el `Trainer` de HuggingFace con los hiperparámetros.

In [ ]:
from transformers import (
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

OUTPUT_DIR = BASE_DIR / "lora_adapter"
NUM_EPOCHS = int(os.getenv("NUM_EPOCHS", "3"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "4"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "2e-4"))
GRADIENT_ACCUMULATION = int(os.getenv("GRADIENT_ACCUMULATION", "4"))

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print(f"Configuración de entrenamiento:")
print(f"  Épocas: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Gradient accumulation: {GRADIENT_ACCUMULATION}")
print(f"  Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  Learning rate: {LEARNING_RATE}")

In [ ]:
# Ejecutar entrenamiento
print("Iniciando entrenamiento LoRA...")
start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print(f"\nEntrenamiento completado en {elapsed:.1f}s")
print(f"Pérdida final: {train_result.training_loss:.4f}")

In [ ]:
# Guardar adaptador
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Adaptador guardado en: {OUTPUT_DIR}")

## 5. Visualizar la loss curve

La loss curve nos indica si el modelo está aprendiendo (loss descendente)
o si hay overfitting (loss de validación sube mientras la de train baja).

In [ ]:
# Extraer logs de entrenamiento
log_history = trainer.state.log_history

train_logs = [log for log in log_history if "loss" in log and "eval_loss" not in log]
eval_logs = [log for log in log_history if "eval_loss" in log]

train_steps = [log["step"] for log in train_logs]
train_loss = [log["loss"] for log in train_logs]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss de entrenamiento
axes[0].plot(train_steps, train_loss, "b-", linewidth=1.5, label="Train Loss")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Loss de validación (si hay)
if eval_logs:
    eval_steps = [log["step"] for log in eval_logs]
    eval_loss = [log["eval_loss"] for log in eval_logs]
    axes[1].plot(eval_steps, eval_loss, "r-o", linewidth=1.5, label="Eval Loss")
    axes[1].set_title("Validation Loss")
else:
    axes[1].plot(train_steps, train_loss, "b-", linewidth=1.5, label="Train Loss")
    axes[1].set_title("Loss (sin validación separada)")

axes[1].set_xlabel("Step")
axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.suptitle(f"LoRA Fine-Tuning — r={LORA_R}, alpha={LORA_ALPHA}", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nPérdida inicial: {train_loss[0]:.4f}")
print(f"Pérdida final:   {train_loss[-1]:.4f}")
print(f"Reducción:       {(1 - train_loss[-1]/train_loss[0])*100:.1f}%")

## 6. Comparar respuestas: antes vs después del fine-tuning

In [ ]:
# Recargar modelo base para comparar
print("Cargando modelo base para comparación...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    token=HUGGINGFACE_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

test_prompts = [
    "Mi cliente recibió el producto dañado y quiere reembolso. ¿Qué hago?",
    "¿Cómo mejoro el ranking de mis productos?",
    "El envío se retrasó 5 días y el cliente está furioso",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"Pregunta: {prompt}")
    print(f"{'='*60}")

    # Respuesta del modelo base
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(base_model.device)
    with torch.no_grad():
        output = base_model.generate(
            input_ids, max_new_tokens=150, temperature=0.7, do_sample=True
        )
    base_response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n[BASE] {base_response.strip()[:200]}")

    # Respuesta del modelo fine-tuneado
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output = model.generate(
            input_ids, max_new_tokens=150, temperature=0.7, do_sample=True
        )
    ft_response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n[LoRA] {ft_response.strip()[:200]}")

## 7. Métricas de calidad

Guardamos las métricas finales para comparar con QLoRA y otras rutas.

In [ ]:
import json

metrics = {
    "train_loss": float(train_result.training_loss),
    "train_runtime_seconds": elapsed,
    "epochs": NUM_EPOCHS,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "model_base": MODEL_BASE,
    "dataset_size": len(train_ds) + len(val_ds),
    "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "total_params": model.num_parameters(),
}

metrics_path = BASE_DIR / "lora_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("Métricas guardadas:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

print(f"\nAdaptador LoRA guardado en: {OUTPUT_DIR}")
print("Siguiente paso: ejecuta 05-bench-paths.ipynb para comparar rutas")

## Resumen

### Qué hicimos
1. Exploramos el dataset de fine-tuning (conversaciones del detective + reseñas)
2. Cargamos el modelo base y aplicamos LoRA (r=16, alpha=32)
3. Entrenamos con el Trainer de HuggingFace PEFT
4. Visualizamos la loss curve y comparamos respuestas antes/después

### Conceptos clave
- **LoRA** inyecta matrices de bajo rank en capas de atención
- Entrena solo ~0.5% de los parámetros, pero captura el estilo del dominio
- El adaptador es un archivo pequeño (~10-100 MB) sobre el modelo base
- **QLoRA** agrega cuantización 4-bit para reducir aún más la VRAM necesaria

### Siguiente paso
→ `05-bench-paths.ipynb`: Comparar base vs prompt vs LoRA vs QLoRA